In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "maqdadali/mental-health-risk-prediction-dataset-students",
    output_dir="/workspace/dataset"
)

print("Dataset downloaded to:", dataset_path)

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_path)

print(df.head())
# print(df.shape) 17 columns
print(df.columns)

In [ ]:
print(df.isnull().sum())
print(df[df.duplicated()])
print(df.info())

In [ ]:
# Age/Year_of_Study/Sleep_Hours/Study_Hours_Per_Week/GPA/Social_Media_Hours
# /Screen_Time_Hours/Financial_Stress/Family_Support/Attendance_Rate       

def detectOutliersIqr(column):
    global df
    if column not in df.columns:
        raise ValueError(f"La colonne '{column}' n'existe pas dans le DataFrame.")
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    filtered_df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    unfiltered_df = df[(df[column] <= lower_bound) | (df[column] >= upper_bound)]
    return [unfiltered_df,df.size-filtered_df.size,Q1,Q3]

# print(detectOutliersIqr('Attendance_Rate'))

In [ ]:
def correctRanges():
    correctDF = df[df['Age'].between(18, 30, inclusive='both')]

    correctDF = correctDF[correctDF['Gender'].isin(['Male', 'Female', 'Non Binary'])]

    correctDF = correctDF[correctDF['Year_of_Study'].between(1, 5, inclusive='both')]

    correctDF = correctDF[correctDF['Sleep_Hours'].between(3.5, 9.5, inclusive='both')]

    correctDF = correctDF[correctDF['Study_Hours_Per_Week'].between(5, 55, inclusive='both')]

    correctDF = correctDF[correctDF['GPA'].between(1.5, 4.0, inclusive='both')]

    correctDF = correctDF[correctDF['Social_Media_Hours'].between(0.5, 10.0, inclusive='both')]

    correctDF = correctDF[correctDF['Screen_Time_Hours'].between(1.0, 14.0, inclusive='both')]

    correctDF = correctDF[correctDF['Physical_Activity'].isin(['Low', 'Moderate', 'High'])]

    correctDF = correctDF[correctDF['Financial_Stress'].between(1.0, 10.0, inclusive='both')]

    correctDF = correctDF[correctDF['Family_Support'].between(1.0, 10.0, inclusive='both')]

    correctDF = correctDF[correctDF['Attendance_Rate'].between(30, 100, inclusive='both')]

    correctDF = correctDF[correctDF['Part_Time_Job'].isin(['Yes', 'No'])]

    correctDF = correctDF[correctDF['Diet_Quality'].isin(['Poor', 'Average', 'Good'])]

    correctDF = correctDF[correctDF['Previous_Mental_Health_History'].isin(['Yes', 'No'])]

    return correctDF


print(len(df))
df=correctRanges()
print(len(df))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='Mental_Health_Risk', data=df)

plt.xlabel('Mental Health Risk')
plt.ylabel('Count')
plt.title('Répartition des classes de Mental Health Risk')
plt.show()

In [ ]:
def repartitionNumerique(column):
    df[column].hist(figsize=(14, 10), bins=20,edgecolor='black')
    plt.tight_layout()
    plt.show()


repartitionNumerique('Attendance_Rate')

In [ ]:
def relationToTargetNumerique(column):
    data0 = df.loc[df['Mental_Health_Risk'] == 'Low', column].dropna()
    data1 = df.loc[df['Mental_Health_Risk'] == 'Medium', column].dropna()
    data2 = df.loc[df['Mental_Health_Risk'] == 'High', column].dropna()

    plt.boxplot(
        [data0, data1, data2],
        tick_labels=['Low', 'Medium', 'High'],
        orientation='horizontal'
    )

    plt.title(f'{column} vs Mental Health Risk')
    plt.xlabel(column)
    plt.ylabel('Mental Health Risk')
    plt.show()

relationToTargetNumerique('Screen_Time_Hours')  

In [ ]:
def relationToTargetCategorielle(column):
    table = pd.crosstab(
        df[column],
        df['Mental_Health_Risk'],
        normalize='index'
    ) * 100

    sns.heatmap(table, annot=True, fmt='.1f')

    plt.title(f'{column} vs Mental Health Risk')
    plt.xlabel('Mental Health Risk')
    plt.ylabel(column)
    plt.show()

relationToTargetCategorielle('Physical_Activity')

In [ ]:
correlation_matrix = df[['Age','Year_of_Study', 'Sleep_Hours','Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours','Screen_Time_Hours','Financial_Stress',
    'Family_Support', 'Attendance_Rate']].corr()


plt.figure(figsize=(12, 8))

sns.heatmap(
        correlation_matrix,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

plt.title("Matrice de correlation")
plt.tight_layout()
plt.show()




In [ ]:
from sklearn.model_selection import train_test_split

X=df[['Age', 'Gender', 'Year_of_Study', 'Sleep_Hours',
       'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours',
       'Screen_Time_Hours', 'Physical_Activity', 'Financial_Stress',
       'Family_Support', 'Attendance_Rate', 'Part_Time_Job', 'Diet_Quality','Previous_Mental_Health_History']]
y=df['Mental_Health_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 , stratify=y
)



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder


numerical_variables = ['Age','Year_of_Study','Sleep_Hours','Study_Hours_Per_Week','GPA','Social_Media_Hours',
    'Screen_Time_Hours','Financial_Stress','Family_Support','Attendance_Rate']

onehot_variables = ['Gender','Part_Time_Job','Previous_Mental_Health_History']

ordinal_variables = ['Diet_Quality','Physical_Activity']

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

onehot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[['Poor', 'Average', 'Good'],['Low', 'Moderate', 'High']]
    ))
])

preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_variables),
    ('onehot', onehot_pipeline, onehot_variables),
    ('ordinal', ordinal_pipeline, ordinal_variables)
])

In [ ]:
X_transformed = preprocessor.fit_transform(X_train)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_transformed = pd.DataFrame(
    X_transformed,
    columns=feature_names,
    index=X_train.index
)

X_transformed.head()

In [ ]:
from sklearn.linear_model import LogisticRegression


base_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=100))
])


base_pipeline.fit(X_train,y_train)
y_pred_base = base_pipeline.predict(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100,random_state=42))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train)
y_test_xgb = label_encoder.transform(y_test)

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100,max_depth=6,learning_rate=0.1,random_state=42))
])

xgb_pipeline.fit(X_train, y_train_xgb)

y_pred_xgb = xgb_pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy_score_base=accuracy_score(y_test,y_pred_base)
precision_score_base=precision_score(y_test,y_pred_base,average='macro')
recall_score_base=recall_score(y_test,y_pred_base, average='macro')
f1_score_base=f1_score(y_test,y_pred_base, average='macro')
confusion_matrix_base=confusion_matrix(y_test,y_pred_base)


accuracy_score_rf=accuracy_score(y_test,y_pred_rf)
precision_score_rf=precision_score(y_test,y_pred_rf,average='macro')
recall_score_rf=recall_score(y_test,y_pred_rf, average='macro')
f1_score_rf=f1_score(y_test,y_pred_rf, average='macro')
confusion_matrix_rf=confusion_matrix(y_test,y_pred_rf)


accuracy_score_xgb=accuracy_score(y_test_xgb,y_pred_xgb)
precision_score_xgb=precision_score(y_test_xgb,y_pred_xgb,average='macro')
recall_score_xgb=recall_score(y_test_xgb,y_pred_xgb, average='macro')
f1_score_xgb=f1_score(y_test_xgb,y_pred_xgb, average='macro')
confusion_matrix_xgb=confusion_matrix(y_test_xgb,y_pred_xgb)




In [ ]:
print(f'base : {accuracy_score_base} | {precision_score_base} | {recall_score_base} | {f1_score_base}')
print(f'rf : {accuracy_score_rf} | {precision_score_rf} | {recall_score_rf} | {f1_score_rf}')
print(f'xgb : {accuracy_score_xgb} | {precision_score_xgb} | {recall_score_xgb} | {f1_score_xgb}')

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Pipeline: Preprocessing + Logistic Regression
logistic_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=2000))
])

# Hyperparameters to tune
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga']
}

# Grid Search with Cross-Validation
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Training
grid_search.fit(X_train, y_train)

# Results
print(f"Best CV score: {grid_search.best_score_:.3f}")
print(f"Best parameters: {grid_search.best_params_}")

# Best complete pipeline
best_model = grid_search.best_estimator_

# Test evaluation
accuracy = best_model.score(X_test, y_test)

print(f"Test set accuracy: {accuracy:.3f}")